# HW3 — Trees & Boosting, Evaluated Properly

**FIN 7057 · Due after Week 7 · 100 pts**

HW2's test period has already informed the research process. This assignment extends the series and
reserves a later forward test window for the comparison of linear, forest, and boosting
models. Part C is the core. A model that fails to beat the linear baseline after costs is a valid
finding when it is reported honestly.


## 0. Setup


In [ ]:
# !pip install -q yfinance scikit-learn scipy   # (xgboost/lightgbm optional)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
SEED=42; np.random.seed(SEED); ANN=np.sqrt(252)
TICKER, START, END = 'SPY', '2010-01-01', '2026-07-01'
HW2_END = pd.Timestamp('2025-01-01')


In [ ]:
def load_prices(t, s, e):
    try:
        import yfinance as yf
        df = yf.download(t, start=s, end=e, progress=False, auto_adjust=True)
        if len(df):
            c = df['Close'].squeeze(); c.name = 'close'; c.index.name = 'date'
            return c.to_frame()
        raise RuntimeError('empty')
    except Exception as ex:
        print(f'synthetic ({ex!r})')
        idx = pd.bdate_range(start=s, end=e, inclusive='left')
        rng = np.random.default_rng(SEED)
        r = rng.standard_normal(len(idx)) * 0.01
        return pd.DataFrame({'close': 100*np.exp(np.cumsum(r))},
                            index=pd.Index(idx, name='date'))

px = load_prices(TICKER, START, END)
px['ret'] = np.log(px['close']).diff()
px = px.dropna()

def spearman_corr(pred, actual):
    pred=np.asarray(pred,float); actual=np.asarray(actual,float)
    return float('nan') if pred.std()==0 or actual.std()==0 else spearmanr(pred,actual).statistic


## Provided: a cost-aware backtest helper
Positions in [-1,1] for day t are applied to the return on t+1. Cost = bps x |position change|.


In [ ]:
def backtest(positions, fwd_returns, cost_bps=0.0):
    """positions, fwd_returns: 1D arrays aligned so positions[t] earns fwd_returns[t].
    Returns dict with gross/net Sharpe, ann. return, turnover, max drawdown."""
    positions=np.asarray(positions,float); fwd=np.asarray(fwd_returns,float)
    gross=positions*fwd
    turnover=np.abs(np.diff(np.concatenate([[0],positions])))
    net=gross - cost_bps/1e4*turnover
    def sh(x): return x.mean()/x.std()*ANN if x.std()>0 else np.nan
    curve=np.cumsum(net); dd=(np.maximum.accumulate(curve)-curve).max()
    return {'gross_Sharpe':sh(gross),'net_Sharpe':sh(net),
            'ann_net_ret':net.mean()*252,'avg_turnover':turnover.mean(),'max_drawdown':dd}


## Part A — Features + three models (time-aware)


In [ ]:
def build_features(prices):
    p=prices.copy(); p['ret']=np.log(p['close']).diff(); f=pd.DataFrame(index=p.index)
    f['ret_lag1']=p['ret'].shift(1)
    for w in (5,21,63): f[f'mom{w}']=p['ret'].rolling(w).mean().shift(1)
    f['vol21']=p['ret'].rolling(21).std().shift(1)
    # TODO: reach >=8 past-only features.
    return f


def assert_feature_future_invariance(make_features, prices, t0_frac=0.6):
    base = make_features(prices)
    pert = prices.copy()
    t0 = int(len(prices)*t0_frac)
    pert.iloc[t0+1:, pert.columns.get_loc('close')] *= 1.25
    changed = make_features(pert)
    d = (base.iloc[:t0+1]-changed.iloc[:t0+1]).abs().to_numpy()
    d = d[np.isfinite(d)]
    return float(d.max()) if d.size else 0.0

feat=build_features(px)
feature_disc=assert_feature_future_invariance(build_features, px)
target=px['ret'].shift(-1)
data=feat.join(target.rename('y')).dropna(); cols=list(feat.columns)

# HW2's assessment ended before HW2_END. Tune only inside the earlier development block,
# then compare models on the newly collected forward test period.
development=data[data.index < HW2_END]
test=data[data.index >= HW2_END]
cut=int(len(development)*0.75)
train,val=development.iloc[:cut],development.iloc[cut:]
Xtr,Xva,Xte=(d[cols].to_numpy() for d in (train,val,test))
ytr,yva,yte=train['y'].values,val['y'].values,test['y'].values
tscv=TimeSeriesSplit(5)

linear=GridSearchCV(
    make_pipeline(StandardScaler(), Ridge()),
    {'ridge__alpha':np.logspace(-4,2,24)}, cv=tscv,
    scoring='neg_mean_squared_error').fit(Xtr,ytr)
rf=RandomForestRegressor(
    n_estimators=400, max_depth=4, min_samples_leaf=50,
    max_features='sqrt', random_state=SEED, n_jobs=-1).fit(Xtr,ytr)
gb_long=GradientBoostingRegressor(
    n_estimators=500,learning_rate=0.03,max_depth=3,
    subsample=0.7,random_state=SEED).fit(Xtr,ytr)
best=int(np.argmin([mean_squared_error(yva,p) for p in gb_long.staged_predict(Xva)]))+1
print(f'boosting early-stop @ {best} trees')
preds={'Linear':linear.predict(Xte),'RandomForest':rf.predict(Xte),
       'GradBoost':list(gb_long.staged_predict(Xte))[best-1]}
print(f'development ends {development.index.max().date()}; '
      f'fresh assessment begins {test.index.min().date()}')


## Part B & C — predictions -> positions -> cost-aware evaluation

Map each next-day-return forecast to a bounded position with expanding, lagged mean and volatility.
The first twenty observations carry zero exposure because the sizing history is not yet available.
This avoids estimating a test-period normalization with future observations.


In [ ]:
fwd_test = yte

def to_positions(pred):
    s=pd.Series(np.asarray(pred,float))
    mu=s.expanding(min_periods=20).mean().shift(1)
    sd=s.expanding(min_periods=20).std().shift(1)
    return np.clip(((s-mu)/(sd+1e-12)).fillna(0.0).to_numpy(),-1,1)

positions={name:to_positions(p) for name,p in preds.items()}
results={}
for name,p in preds.items():
    pos=positions[name]
    row={'test_R2':r2_score(yte,p),'test_Spearman':spearman_corr(p,yte)}
    for bps in (0,5,10,20):
        bt=backtest(pos,fwd_test,cost_bps=bps)
        row[f'net_Sharpe_{bps}bps']=round(bt['net_Sharpe'],2)
    row['turnover']=round(backtest(pos,fwd_test)['avg_turnover'],2)
    results[name]=row
print(pd.DataFrame(results).T.round(3))
# TODO: report max drawdown at a stated realistic cost and justify that cost.


## Part D — comparison table & verdict

TODO (>=200 words): Compare test R², test Spearman correlation, gross Sharpe, net Sharpe, turnover,
and buy-and-hold. Did either nonlinear model beat the linear baseline after costs? If not, say so.
Which model, if any, would you deploy and why?


In [ ]:
bh=fwd_test
print(f'Buy-and-hold assessment Sharpe: {bh.mean()/bh.std()*ANN:+.2f}')
# TODO: assemble the final table with test R², test Spearman, gross/net Sharpe, and turnover.


## Part E — multiple testing & limitations

TODO:
1. Estimate the number of configurations you tried.
2. State the assumptions behind an approximate expected-best null Sharpe or another multiplicity
   adjustment. Do not present the approximation as a precise deflated-Sharpe statistic.
3. Discuss capacity, post-publication decay, regime dependence, and why the assessment is now spent.


## Extensions (optional)

The required core is everything above, and the self-check tests only that. These extensions are optional and **not** required for full marks — a clean, well-reasoned core beats a sprawling one. If you want to go further:

- Run a small hyperparameter search (learning rate x depth) with time-aware early stopping.
- Compare a random forest against the boosting model on the same out-of-sample, cost-aware metric.
- Add permutation importance -- and state why it is not a causal mechanism.

## Self-check (auto-graded)

Run this cell **last**, after the whole notebook. Every line must print **PASS** before you submit — it verifies the mechanical requirements (reproducibility, leak-free features, time-ordered splits, metrics computed). Your written answers and judgment are graded by hand.

In [ ]:
# ================= SELF-CHECK (auto-graded) =================
def _check(name, fn, hint=""):
    try:
        ok=bool(fn())
    except Exception as e:
        ok,hint=False,hint or f"({type(e).__name__}: {e})"
    print(("PASS  " if ok else "FAIL  ")+name+("" if ok else "   -> "+hint))
    return ok

def _position_future_disc():
    p=np.asarray(next(iter(preds.values())),float)
    base=to_positions(p); changed=p.copy(); t0=len(p)//2
    changed[t0+1:]=changed[t0+1:]*-3+0.1
    return float(np.max(np.abs(base[:t0+1]-to_positions(changed)[:t0+1])))

_r=[]
_r.append(_check("reproducibility: SEED set",lambda:SEED is not None,"set SEED"))
_r.append(_check(">= 8 features",lambda:feat.shape[1]>=8,"reach >=8 past-only features"))
_r.append(_check("future perturbation leaves past features unchanged",lambda:feature_disc<1e-8,"remove look-ahead from build_features"))
_r.append(_check("fresh assessment follows HW2",lambda:test.index.min()>=HW2_END and development.index.max()<test.index.min(),"reserve observations after HW2_END"))
_r.append(_check("train, validation, assessment are disjoint",lambda:train.index.max()<val.index.min() and val.index.max()<test.index.min(),"enforce train < val < assessment"))
_r.append(_check("position scaling is past-only",lambda:_position_future_disc()<1e-10,"use lagged expanding/rolling sizing statistics"))
_r.append(_check("forecast metrics computed",lambda:all(np.isfinite(v['test_R2']) and np.isfinite(v['test_Spearman']) for v in results.values()),"report R2 and single-series Spearman"))
_r.append(_check("backtest returns cost-aware metrics",lambda:{'net_Sharpe','avg_turnover','max_drawdown'}<=set(backtest(np.zeros(20),np.zeros(20)).keys()),"keep backtest()"))
print(f"\n{sum(_r)}/{len(_r)} mechanical checks passed.")
assert all(_r),"Fix the FAIL items above before submitting."


## AI-use disclosure (required)
TODO.

## Reproducibility checklist
- [ ] Restart & Run All  - [ ] seed  - [ ] dates  - [ ] source  - [ ] packages
